# Weather Across the Season at Petco Park

**Goal:** Visualize how temperature, pressure, humidity, and wind speed vary by month at Petco Park (San Diego Padres) to understand seasonal weather patterns and their potential effects on run scoring.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load league-wide game data and filter to SD home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
sd = data[data['home_team'] == 'SD'].copy()

# Parse game_date to extract month
sd['game_date'] = pd.to_datetime(sd['game_date'])
sd['month'] = sd['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
             7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
sd['month_name'] = sd['month'].map(month_map)

months = sorted(sd['month'].unique())
month_names = [month_map.get(m, str(m)) for m in months]

print(f"Total SD home games: {len(sd)}")
print(f"\nGames per month:")
print(sd.groupby('month_name').size().reindex([month_map[m] for m in months]))

In [ ]:
# ============================================================
# Weather Distributions by Month
# ============================================================

weather_vars = {
    'temp_f':   {'label': 'Temperature (°F)', 'color': '#d62728'},
    'rhum':     {'label': 'Humidity (%)',      'color': '#1f77b4'},
    'wspd_mph': {'label': 'Wind Speed (mph)',  'color': '#2ca02c'},
    'pres':     {'label': 'Pressure (hPa)',    'color': '#9467bd'},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, info) in zip(axes.flat, weather_vars.items()):
    box_data = [sd.loc[sd['month'] == m, col].dropna().values for m in months]
    bp = ax.boxplot(box_data, patch_artist=True, labels=month_names,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor(info['color'])
        patch.set_alpha(0.6)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel(info['label'], fontsize=11)
    ax.set_title(info['label'], fontsize=13)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle('Weather Distributions by Month \u2014 Petco Park', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Scatter Plots: Temperature vs. Pressure & Temperature vs. Humidity
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Temperature vs. Pressure
ax = axes[0]
scatter = ax.scatter(sd['temp_f'], sd['pres'], c=sd['month'], cmap='coolwarm',
                     alpha=0.6, edgecolors='black', linewidth=0.3, s=40)
z = np.polyfit(sd['temp_f'].dropna(), sd.loc[sd['temp_f'].notna(), 'pres'], 1)
p = np.poly1d(z)
x_line = np.linspace(sd['temp_f'].min(), sd['temp_f'].max(), 100)
ax.plot(x_line, p(x_line), color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('Temperature (\u00b0F)', fontsize=11)
ax.set_ylabel('Pressure (hPa)', fontsize=11)
ax.set_title('Temperature vs. Pressure', fontsize=13)
ax.yaxis.grid(True, alpha=0.3)
ax.xaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Temperature vs. Humidity
ax = axes[1]
scatter = ax.scatter(sd['temp_f'], sd['rhum'], c=sd['month'], cmap='coolwarm',
                     alpha=0.6, edgecolors='black', linewidth=0.3, s=40)
z = np.polyfit(sd['temp_f'].dropna(), sd.loc[sd['temp_f'].notna(), 'rhum'], 1)
p = np.poly1d(z)
ax.plot(x_line, p(x_line), color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('Temperature (\u00b0F)', fontsize=11)
ax.set_ylabel('Humidity (%)', fontsize=11)
ax.set_title('Temperature vs. Humidity', fontsize=13)
ax.yaxis.grid(True, alpha=0.3)
ax.xaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

cbar = fig.colorbar(scatter, ax=axes, label='Month', ticks=months)
cbar.ax.set_yticklabels(month_names)

fig.suptitle('Weather Relationships \u2014 Petco Park', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Day vs. Night classification
# ============================================================

# Day games start before 5 PM, night games at 5 PM or later
sd['time_of_day'] = sd['start_hour'].apply(lambda h: 'Day' if h < 17 else 'Night')

print("Game counts by month and time of day:")
print(sd.groupby(['month_name', 'time_of_day']).size().unstack(fill_value=0)
      .reindex([month_map[m] for m in months]))

In [ ]:
# ============================================================
# Average Temperature: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = sd[sd['time_of_day'] == label]
    grouped = subset.groupby('month')['temp_f']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.3,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Temperature (\u00b0F)', fontsize=12)
ax.set_title('Average Temperature: Day vs. Night Games by Month \u2014 Petco Park', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Humidity: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = sd[sd['time_of_day'] == label]
    grouped = subset.groupby('month')['rhum']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.3,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Humidity (%)', fontsize=12)
ax.set_title('Average Humidity: Day vs. Night Games by Month \u2014 Petco Park', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Pressure: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

all_means = []
all_sems = []
for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = sd[sd['time_of_day'] == label]
    grouped = subset.groupby('month')['pres']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    all_means.append(means)
    all_sems.append(sems)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.1,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

# Zoom y-axis to better show differences
y_min = min(m.min() for m in all_means) - 2
y_max = max(m.max() + s.max() for m, s in zip(all_means, all_sems)) + 1.5
ax.set_ylim(y_min, y_max)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Pressure (hPa)', fontsize=12)
ax.set_title('Average Pressure: Day vs. Night Games by Month \u2014 Petco Park', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Total Runs: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = sd[sd['time_of_day'] == label]
    grouped = subset.groupby('month')['total_runs']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.1,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Runs', fontsize=12)
ax.set_title('Average Total Runs: Day vs. Night Games by Month \u2014 Petco Park', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Wind Direction on High Wind Days: Day vs. Night
# ============================================================

# Top quintile of wind speed
sd['wspd_bin'] = pd.qcut(sd['wspd_mph'], q=5, duplicates='drop')
top_bin = sd['wspd_bin'].cat.categories[-1]
high_wind = sd[sd['wspd_bin'] == top_bin]

compass = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, color) in zip(axes, [('Day', '#ff9900'), ('Night', '#003366')]):
    subset = high_wind[high_wind['time_of_day'] == label]
    dir_counts = subset['wind_dir_bucket'].value_counts().reindex(compass, fill_value=0)
    ax.bar(dir_counts.index, dir_counts.values, color=color, alpha=0.75,
           edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Wind Direction (blowing from)', fontsize=11)
    ax.set_ylabel('Number of Games', fontsize=11)
    ax.set_title(f'{label} Games (n={len(subset)})', fontsize=13)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    for xi, v in enumerate(dir_counts.values):
        ax.text(xi, v + 0.3, str(v), ha='center', va='bottom', fontsize=10)

fig.suptitle(f'Wind Direction on High Wind Days (Top Quintile: {top_bin})\nDay vs. Night \u2014 Petco Park',
             fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary Statistics Table
# ============================================================

summary = sd.groupby('month').agg(
    games=('temp_f', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    rhum_mean=('rhum', 'mean'),
    rhum_std=('rhum', 'std'),
    wspd_mean=('wspd_mph', 'mean'),
    wspd_std=('wspd_mph', 'std'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
).round(2)

summary.index = [month_map.get(m, str(m)) for m in summary.index]
summary.index.name = 'Month'
summary.columns = ['Games', 'Temp Mean (\u00b0F)', 'Temp Std',
                    'Humidity Mean (%)', 'Humidity Std',
                    'Wind Mean (mph)', 'Wind Std',
                    'Pressure Mean (hPa)', 'Pressure Std']
summary